# Streaming

Streaming reduces the latency between generating data and the user receiving it. There are two types frequently used with Agents:

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model = "ollama:qwen2.5:3b",
    system_prompt="You are a full-stack comedian"
)

## No Streaming (invoke)

In [3]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][-1].content)

Sure, here's one for you:

Why don't scientists trust atoms?

Because they make up everything!


# values
You have seen this streaming mode in our examples so far.

In [4]:
#Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a dank joke"}]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a dank joke
================================== Ai Message ==================================

Sure thing, friend! Here’s one for you:

Why don't skeletons ever get lost? Because they have all their bones tied together like knots in a rope!

Now, let's hope I didn't knot your brain with that one. But hey, it's fun and knot-too-badly delivered!


# Messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [5]:
for token, metadata in agent.stream(
    {"messages": {"role":"user", "content": "Write a family friendly poem."}},
    stream_mode="messages"
):
    print(f"{token.content}", end="")

In the center of my garden, where bees dance with flowers,
Lives a little boy named Tommy, and he's in love with his toys.
With cars that can zoom and balls that can bounce,
Tommy plays all day long under a sky blue as soap.

He has a treehouse where stories are told,
And every evening he goes to bed on time.
His parents read him bedtime rhymes,
A soothing lullaby, from the first till the last rhyme.

Their kitchen is his secret hideaway,
Where his mom bakes cookies for them all day long.
Tommy gets to taste them before they're served,
With a smile that makes everyone's heart so bright and cared for.

So whether he’s building with blocks or eating chocolate chip cookies,
His family knows Tommy’s happiness can be found in simple pleasures like these.
For each little joy, his days are filled with love and care.

# Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer writer` allows you to easily stream `custom`data from sources you create.

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
   print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='9dc13ba7-7a56-4c83-9a9b-59ffb5bdf9c1')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='9dc13ba7-7a56-4c83-9a9b-59ffb5bdf9c1'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-10T16:50:28.525455Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1056799833, 'load_duration': 152060083, 'prompt_eval_count': 152, 'prompt_eval_duration': 70082000, 'eval_count': 20, 'eval_duration': 822315000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f4cf0-0a0a-7320-aeaa-ef484c067447-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'd97a9e47-0f2d-4ee0-b6d4-63a8e29ffa3f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 20, 

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')
